# Manifold Learning: t-SNE

## Science Question: Can we separate OIB samples by mantle sources?
Ocean island basalts (OIBs) are igneous rocks that show variability in geochemical composition, often linked to the underlying variability in mantle plume sources. Therefore, they provide a source for investigating the composition of the Earth's deep mantle (from upwelling plumes), as opposed to the mid-ocean-ridge basalts (MORB) that are produced by the well-mixed, depleted mantle. A few decades of radiogenic isotope measurements (Sr, Nd, Pb, Hf) from OIBs have shown that their isotopic compositions do not form one single cluster, but instead separate into distinct groups, informally known as the mantle "zoo" (Zindler and Hart, 1986):

- **DMM** : Depleted MORB Mantle reservoir sampled by mid-ocean ridges, with high $^{143}\text{Nd}/^{144}\text{Nd}$ with variability depending on the source, i.e., Atlantic, Indian, or Pacific Oceans,

- **HIMU** : mantle source with a high time-integrated U/Pb ratio ($\mu = {}^{238}\text{U}/{}^{204}\text{Pb}$), characterized by very radiogenic Pb (high $^{206}\text{Pb}/^{204}\text{Pb}$), low $^{87}\text{Sr}/^{86}\text{Sr}$, among other element ratios, observed in some OIBs (e.g., St. Helena, Cook-Austral islands),

- **EM1** : Enriched Mantle 1, with lower $\text{U}/\text{Pb}$ and lower $^{206}\text{Pb}/^{204}\text{Pb}$ than HIMU, intermediate $^{87}\text{Sr}/^{86}\text{Sr}$ values, distinct from EM2 and often attributed to recycled ancient oceanic crust materials observed in some OIBs (e.g., Kerguelen, Tristan),

- **EM2** : Enriched Mantle 2, with higher $^{206}\text{Pb}/^{204}\text{Pb}$ than EM1 and $^{87}\text{Sr}/^{86}\text{Sr}$ values, often attributed to deep mantle recycling of subducted crust/sediments, observed in some OIBs (e.g., Samoa, Society Islands),

These components are classically identified using isotope diagrams between two variables, for example $^{87}\text{Sr}/^{86}\text{Sr}$ versus $^{206}\text{Pb}/^{204}\text{Pb}$, to identify a cluster of points that can be attributed to a particular mantle source. However, with multiple isotope ratio measurements, it is not always clear which combination of isotope ratios is best to separate the different mantle sources. In this notebook, we will use **T-distributed Stochastic Neighbor Embedding** (t-SNE) to visualize the OIB data in a lower-dimensional space and see if we can separate the samples by mantle source.

As with PCA, t-SNE is a machine learning technique to reduce the dimensionality of complex data, in order to help identify patterns and underlying petological processes (recycled crust, delaminated lithosphere, primordial material, etc.) and simplify the visualization in the lower-dimension space.

## t-SNE technique
PCA (see the [previous notebook](1_pca_geochemical_data.ipynb)) finds new axes, i.e., eigenvectors, that are linear combinations of the original features. This works well when the dominant structure in the data is linear, but it can fail when data is curved or made up of several separate clusters, because linear directions cannot capture that kind of shape.

**Manifold learning** methods, including t-SNE, flattens or unrolls the high-dimensional data to create a lower-dimensional representation while preserving the *local neighborhood structure* of the data: points that are close together in the original, high-dimensional feature space should remain close together in the low-dimensional map.

<div align="center">
    <img src='./images/manifold_learning.png' width="400"/>
    <figcaption> Data (grey dots) approximation using a manifold learning technique using a 1D self-organizing map (SOM, red) along with the first principal component (blue).  Image: Wikipedia </figcaption>
</div>

t-SNE represents similarities between data points in the original high-dimensional space as probabilities, and projects them into a lower-dimensional space by minimizing the difference between the two probability distributions. The algorithm has a few hyperparameters that can be tuned, including the **perplexity**, which is related to the number of nearest neighbors considered when computing the probabilities. A low perplexity value will focus on local structure, while a high perplexity value will consider more global structure.

<div align="center">
    <img src='./images/tsne_diagram.png' width="500"/>
    <figcaption> The original high-dimensional data (left) is converted to Gaussian joint probability distribution in the low-dimensional space (right). Image <a href="https://spotintelligence.com/2023/12/22/practical-guide-t-sne/" target="_blank">source</a> </figcaption>
</div>


### Problem setup

Step 1. **In the original data, turn distances into neighbor probabilities**

Suppose we have $n$ samples described by $p$ features (here, isotope ratios). For every pair of points $i,j$ in the high-dimensional space, t-SNE defines a conditional probability that point $i$ would pick point $j$ as its neighbor, using a Gaussian kernel centered on $x_i$:

$$
p_{j|i} = \frac{\exp\left(-\lVert x_i - x_j \rVert^2 / 2\sigma_i^2\right)}{\sum_{k \neq i} \exp\left(-\lVert x_i - x_k \rVert^2 / 2\sigma_i^2 \right)}
$$

We can understand the above equation as: the numerator computes distance between points as neighbor probabilities, while the denominator normalizes the probabilities so that they sum to 1. The bandwidth $\sigma_i$ of the Gaussian kernel controls how wide a neighborhood point sees, and is the perplexity hyperparameter of t-SNE.
The joint probability is then defined as $p_{ij} = (p_{j\mid i} + p_{i \mid j})/2n$, which describes the *neighborhood structure* of the data in the original feature space. t-SNE then seeks to find a low-dimensional representation of the data that best preserves this neighborhood structure.

Step 2. **On the map, define a new kind of similarity**

If each sample is described by position $y_i$ in the low-dimensional space, the similarity between the two points $y_i$ and $y_j$ is defined using another distance probability, q_{ij}:

$$
q_{ij} = \frac{\left(1 + \lVert y_i - y_j \rVert^2\right)^{-1}}{\sum_k \sum_{k \neq l} \left(1 + \lVert y_k - y_l \rVert^2\right)^{-1}}
$$

The above equation is called the Student-t distribution. It lets moderately distant points in the high-dimensional space be placed further apart in the map than a Gaussian would allow, which helps prevent crowding and makes it easier to see separate clusters.

Step 3. **Minimize the difference between the two distributions**

The map coordinates $y_i$ are then found by minimizing, via gradient descent, the difference between the high-dimensional neighborhoods $P$ and the low-dimensional neighborhoods $Q$ in log space:

$$
f(P \parallel Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}
$$

The equation above is non-unique, meaning that the final map can be rotated or flipped and still represent the same neighborhood structure. Therefore, t-SNE is often run multiple times depending on the initialization, and the map with the lowest cost function (i.e., the lowest value of $f(P \parallel Q)$) is chosen.

**Important information about reading t-SNE plots**:
 - Distance between different clusters and the sizes of clusters, are not generally meaningful; only local neighborhoods are approximately preserved.
 - The axes of a t-SNE plot have no direct physical meaning, unlike PCA loadings.

## t-SNE in python

We will use `pandas` and `scikit-learn` to perform t-SNE on isotope-ratio data downloaded from [PetDB](https://www.earthchem.org/petdb).

Particularly, we will:

- inspect the raw download and identify the relevant radiogenic isotope ratio columns and
- clean data to retain rows with complete isotope measurements.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from scipy.spatial.distance import cdist

plt.rcParams["figure.figsize"] = (6, 5)


### 1. Inspect the data

As in the PCA notebook, the raw PetDB export contains one row per analyzed sample, with metadata columns (rock type, tectonic setting, location, age, etc.) and isotope ratios in columns named, e.g., `SR87_SR86` or `PB206_PB204`.

In [ ]:
# Load the raw PetDB export.
# Some data is string while some is numeric, so we use the `low_memory=False` option to avoid dtype warnings.
input_geochem_data = pd.read_csv("data/earthchem_download_sne.txt", sep="\t", low_memory=False)

# Some column names also have trailing whitespace (e.g. "SR87_SR86 "), so we strip the column names as well.
input_geochem_data.columns = input_geochem_data.columns.str.strip()

print(f"Raw data: {input_geochem_data.shape[0]} samples, {input_geochem_data.shape[1]} columns")

# Check how many samples we have in each of the unique tectonic setting
input_geochem_data["TECTONIC SETTING"].value_counts()


In [ ]:
# We choose some of the isotope ratios that are commonly used in geochemistry
# and based on figure 2 by White et al. (2025).
candidate_isotopes = ["SR87_SR86", "PB206_PB204", "PB208_PB204", "ND143_ND144", "PB207_PB204"]
feature_names = [c for c in candidate_isotopes if c in input_geochem_data.columns]

isotope = input_geochem_data[feature_names]
print("Fraction of all samples with a reported value:\n", isotope.notna().mean())


### 2. Clean data: retrieve samples with complete isotope data
As in the PCA notebook, a missing isotope measurement does not mean the ratio is zero, so we keep only rows with a complete set of the five isotope ratios above.

In [ ]:
analysis_df = (input_geochem_data
               .dropna(subset=feature_names))

print(f"Samples with complete isotope data: {len(analysis_df)}")

# look at the values of the isotope ratios for the samples with complete data
analysis_df[feature_names].describe().T[["mean", "std", "min", "max"]]


A mean value above for $^{143}\text{Nd}/^{144}\text{Nd}$ sits far outside the physically possible range for terrestrial rocks (natural values are close to 0.511-0.514, see figure 2 in Strake et al., 2005), and is most likely an error in data entry. We drop rows with isotope ratios outside the expected physically reasonable bounds before continuing.

In [ ]:
plausible = (analysis_df["ND143_ND144"].between(0.505, 0.515))
print(f"Dropping {(~plausible).sum()} sample(s) with implausible isotope ratios")

analysis_df = analysis_df[plausible]
print(f"Remaining samples for analysis: {len(analysis_df)}")


### 3. Assign approximate mantle-component labels

Unlike the `TECTONIC SETTING` or `ROCK TYPE` columns used in the PCA notebook, we do not have a mantle-component label (i.e., EM1, EM2, HIMU, DMM) attached to each sample. To visualize whether samples with different likely mantle compositions separate in the t-SNE map, we first build an approximate label ourselves:

1. We define idealized isotopic compositions for DMM, HIMU, EM1, and EM2, compiled from widely cited mantle-geochemistry reviews (Zindler and Hart, 1986; White, 2010; Stracke, 2012).
2. We standardize the samples and reference compositions using the same scaling for comparison, so that the different isotope ratios are comparable in the same space.
3. We assign each sample to its **nearest reference composition** (in standardized isotope space).

> This is a simplified classification for the purposes of this tutorial, not a rigorous petrogenetic determination. Real mantle end-members are not sharply bounded points but broad, overlapping, debated regions in isotope space

In [ ]:
# Idealized mantle end-member isotope compositions taken from Table 2 in Jackson and Dasgupta (2008)
# and Figure 5 in Workmann et al., 2003.
reference_components = pd.DataFrame(
    {
        "SR87_SR86":   [0.7028, 0.7084, 0.7046, 0.7023],
        "ND143_ND144": [0.5133, 0.5129, 0.5123, 0.5136],
        "PB206_PB204": [21.199, 19.149, 17.826, 18.390],
        "PB207_PB204": [15.767, 15.623, 15.496, 15.506],
        "PB208_PB204": [40.373, 39.444, 38.823, 37.840],
    },
    index=["HIMU", "EM2", "EM1", "DMM"],
)[feature_names]

scaler = StandardScaler()

# We standardize the samples using `fit_transform`, which computes mean and standard deviation,
# for the given data, and then use the same scaling to standardize the reference compositions
# using `transform`.
sample_scaled = scaler.fit_transform(analysis_df[feature_names])
reference_z = scaler.transform(reference_components)

# sample_scaled is of size (n_samples, n_features) and reference_z is of size (n_mantle_components, n_features).
# What we want is to compute distance between each sample and each reference composition,
# which will give us a distance matrix of size (n_samples, n_mantle_components). `cdist`
# computes the distance between each pair of the two collections of inputs.
distances = cdist(sample_scaled, reference_z)
analysis_df = analysis_df.copy()

# Assign mantle component to each sample based on the minimum distance to the reference compositions.
analysis_df["mantle_component"] = reference_components.index[np.argmin(distances, axis=1)]

analysis_df["mantle_component"].value_counts()


### 4. Plot isotope ratios

Before reducing the dimensionality, we plot the isotope ratios of samples based on how it is done in other studies: a scatter plot of $^{87}\text{Sr}/^{86}\text{Sr}$ against $^{206}\text{Pb}/^{204}\text{Pb}$, colored by the approximate mantle-component label.

In [ ]:
components = reference_components.index.tolist()
colors = dict(zip(components, plt.cm.tab10.colors))

fig, ax = plt.subplots(figsize=(6, 5))
for component in components:
    mask = (analysis_df["mantle_component"] == component).to_numpy()
    ax.scatter(
        analysis_df.loc[mask, "SR87_SR86"],
        analysis_df.loc[mask, "PB206_PB204"],
        s=35, alpha=0.7, color=colors[component], label=f"{component} (n={mask.sum()})")

# We also plot the idealized reference compositions
for component in components:
    ax.scatter(reference_components.loc[component, "SR87_SR86"], reference_components.loc[component, "PB206_PB204"],
        marker="*", s=300, color=colors[component], edgecolor="black", zorder=5,
    )

ax.set_xlabel(r"$^{87}$Sr/$^{86}$Sr")
ax.set_ylabel(r"$^{206}$Pb/$^{204}$Pb")
ax.set_title("Sr-Pb isotope diagram")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
plt.show()


In the above figure, the star represents the idealized reference composition for each mantle component, while the points represent the OIB samples. The separation of the clusters is not perfect, especially for EM1 and EM2 in this two-isotope diagram. Next, we use t-SNE to see whether including all five isotope ratios at once separates the samples better by mantle source.

### 5. Fitting t-SNE

We convert the five dimensions of standardized isotope ratios into a two-dimensional t-SNE map, governed by `n_components`.

In [ ]:
# The starting point for the t-SNE model is the pca values.
tsne = TSNE(
    n_components=2,
    perplexity=10,
    init="pca",
    learning_rate="auto",
    random_state=42,
)
embedding = tsne.fit_transform(sample_scaled)

fig, ax = plt.subplots(figsize=(6, 5))
for component in components:
    mask = (analysis_df["mantle_component"] == component).to_numpy()
    ax.scatter(embedding[mask, 0], embedding[mask, 1], s=40, alpha=0.75,
               color=colors[component], label=f"{component} (n={mask.sum()})")

ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
ax.set_title("t-SNE embedding of isotope ratios (perplexity = 10)")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout(rect=[0, 0, 0.78, 1])
plt.show()


The results from t-SNE are similar to the distance-based separation of the samples in the figure above, with the EM1 and EM2 samples overlapping in the t-SNE map. This makes sense, because the two isotope ratios used in the figure above are the most important for separating mantle components, and t-SNE conserves these relationships in the low-dimensional space.

The DMM and HIMU samples are more clearly separated from the other two components, but HIMU forms a second tight cluster.

### 6. Effect of perplexity

Because perplexity effectively sets the neighborhood size used to build the low-dimensional similarities, the results may change noticeably with different perplexity values.

In [ ]:
perplexities = [5, 10, 20, 30]

fig, axes = plt.subplots(1, len(perplexities), figsize=(5 * len(perplexities), 4.5), sharex=False, sharey=False)
for ax, perplexity in zip(axes, perplexities):
    emb = TSNE(
        n_components=2, perplexity=perplexity, init="pca",
        learning_rate="auto", random_state=42,
    ).fit_transform(sample_scaled)

    for component in components:
        mask = (analysis_df["mantle_component"] == component).to_numpy()
        ax.scatter(emb[mask, 0], emb[mask, 1], s=25, alpha=0.75, color=colors[component])
    ax.set_title(f"perplexity = {perplexity}")
    ax.set_xlabel("t-SNE dimension 1")
    ax.set_ylabel("t-SNE dimension 2")

handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=colors[c], markersize=8, label=c)
           for c in components]
fig.legend(loc="lower center", handles=handles, ncol=len(components), bbox_to_anchor=(0.5, -0.05))

plt.show()


The plot above illustrates:

- results of t-SNE are quite sensitive to the perplexity hyperparameter, and the overall shape, spacing, and even relative position of clusters can change noticeably with perplexity
- DMM is always separated clearly from the other 3 components.
- HIMU is split into two clusters, which is different from what we see in the published isotope diagrams where HIMU is often cleanly separated from other components (see figure 2-3 in Stracke et al., 2004). This may be an artifact of how we assigned the mantle component labels or it may reflect the fact that HIMU has distinct sub-components in the high-dimensional space that are not obvious in 2-D isotope diagrams.
- EM1 is always mixed with EM2, which could be due to the lack of enough samples in the dataset to fully capture the EM1 and EM2 clusters, or it could reflect the intrinsic similarity of the two components in the high-dimensional isotope space.

## Try it yourself

- Refit t-SNE with a different `random_state`. Do the same samples still cluster together, even though the exact layout changes?

- Compare the t-SNE embedding above to a PCA projection of the same five standardized isotope ratios (see `1_pca_geochemical_data.ipynb` for the PCA workflow). Does PCA separate the mantle components as well as t-SNE does?

- Our reference end-member compositions are approximate. Try adjusting them using values from one of the references below, or from a paper describing a specific OIB locality (e.g. Pitcairn for EM1, Samoa for EM2, Mangaia or St. Helena for HIMU), and see how the assigned labels change.

### References
- Zindler, A., and Hart, S. (1986). Chemical Geodynamics. *Annual Review of Earth and Planetary Sciences*, [14, 493-571](https://doi.org/10.1146/annurev.ea.14.050186.002425).
- Workman, R. K., Hart, S. R., Jackson, M., Regelous, M., Farley, K. A., Blusztajn, J., ... & Staudigel, H. (2004). Recycled metasomatized lithosphere as the origin of the Enriched Mantle II (EM2) end‐member: Evidence from the Samoan Volcanic Chain. Geochemistry, Geophysics, Geosystems, [5(4)](10.1029/2003GC000623).
- Stracke, A., Hofmann, A. W., & Hart, S. R. (2005). FOZO, HIMU, and the rest of the mantle zoo. Geochemistry, geophysics, geosystems, [6(5)](https://doi.org/10.1029/2004GC000824).
- Jackson, M. G., & Dasgupta, R. (2008). Compositions of HIMU, EM1, and EM2 from global trends between radiogenic isotopes and major elements in ocean island basalts. Earth and Planetary Science Letters, 276(1-2), [175-186](https://doi.org/10.1016/j.epsl.2008.09.023).
- White, W. M., Jackson, M. G., & Hardardottir, S. (2025). Insights into mantle plume geochemistry from machine learning. Geochemistry, Geophysics, Geosystems, 26(1), [e2024GC011870](https://doi.org/10.1029/2024GC011870).
- The data was downloaded from the [PetDB Database](https://www.earthchem.org/petdb) using the tectonic setting = ocean basin, spreading center, and forearc basins.

### Further Reading
- Other manifold-learning methods, such as UMAP ([Uniform Manifold Approximation and Projection](https://umap-learn.readthedocs.io/en/latest/supervised.html)) and LLE ([Locally Linear Embedding](https://www.science.org/doi/10.1126/science.290.5500.2323)) use related ideas (preserving local neighborhoods) but optimize a different objective and often run faster on larger datasets.